# Modelos y modelo propuesto: Ridge jerárquico con nivel adaptativo

Este cuaderno corre el experimento completo del artículo sobre la base de Colombia (`tabla_diaria_colombia.csv`, que arma
`Clima_a_Generacion_Colombia.ipynb`). Se predice el **factor de capacidad (FC) diario** de cada planta, como
variación frente a su nivel habitual. El cuaderno compara modelos de regresión de referencia con el modelo propuesto, hace las
pruebas estadísticas, la ablación y las pruebas de sobreajuste.

Protocolo: cada planta se parte en el tiempo (primera mitad de sus días = pasado, segunda mitad = prueba) y en cada pliegue externo
se deja fuera una zona completa de plantas cercanas. Los hiperparámetros se eligen con validación anidada.

Los resultados se guardan en `paper/resultados/` y las figuras del artículo se generan con `paper/figures/make_figures.py`.

## 1. Datos, partición y centrado

La partición va primero. El nivel de cada planta y la media de sus covariables salen solo de su pasado.

In [1]:
import json, os, platform, time, warnings
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
import xgboost
from joblib import Parallel, delayed
from scipy import stats
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import GridSearchCV, LeaveOneGroupOut, KFold
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
SEED = 42
RES = Path("../paper/resultados")
RES.mkdir(parents=True, exist_ok=True)

tabla = pd.read_csv("../data/tabla_diaria_colombia.csv", parse_dates=["date"]).sort_values(["code", "date"]).reset_index(drop=True)
VARS = ["rad_bm", "nube_bm", "temp_bm", "humedad_bm", "viento_bm"]
VARS_TODAS = VARS + ["rad_e5", "nube_e5", "e_fis_bm"]
tabla["prueba_t"] = tabla.groupby("code").cumcount() >= tabla.groupby("code").fc.transform("size") // 2
PASADO = (~tabla.prueba_t).values


def prepara(datos, filas_nivel):
    """Centra el FC y las covariables con el nivel de cada planta calculado SOLO con `filas_nivel` (su pasado)."""
    medias = datos[filas_nivel].groupby("code")[VARS_TODAS + ["fc"]].mean()
    d = datos.copy()
    for v in VARS_TODAS + ["fc"]:
        d[v + "_c"] = d[v] - d.code.map(medias[v])
    # FC del día anterior, centrado con el nivel de la planta (sin dato, se usa el nivel): persistencia y comparación justa
    d["fcayer_c"] = (d.fc_ayer - d.code.map(medias["fc"])).fillna(0.0)
    return d


def particiones_zona():
    for z in sorted(tabla.grupo.unique()):
        ent = (~tabla.prueba_t) & (tabla.grupo != z)   # el modelo aprende solo del pasado de otras zonas
        pru = tabla.prueba_t & (tabla.grupo == z)       # y se evalúa en el futuro de la zona que dejó fuera
        yield z, ent.values, pru.values


D = prepara(tabla, PASADO)
print(tabla.shape, "| plantas:", tabla.code.nunique(), "| zonas:", tabla.grupo.nunique(),
      "| filas de prueba:", int(tabla.prueba_t.sum()))

(8589, 24) | plantas: 16 | zonas: 6 | filas de prueba: 4300


## 2. Modelos de referencia

Modelos de regresión supervisada de referencia: regresión lineal, Ridge, Lasso, KNN, SVR, Kernel Ridge, Random Forest y
XGBoost. Cada uno se evalúa con sus valores por defecto y con búsqueda en rejilla dentro de la validación anidada.

In [2]:
pipe = lambda m: Pipeline([("esc", StandardScaler()), ("m", m)])
REG = {
    "Referencia: nivel de la planta": DummyRegressor(strategy="constant", constant=0.0),
    "Regresión lineal": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.001, max_iter=10000),
    "KNN": KNeighborsRegressor(n_neighbors=25),
    "SVR RBF": SVR(C=1.0, epsilon=0.01),
    "Kernel Ridge RBF": KernelRidge(kernel="rbf", alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=1),
    "XGBoost": XGBRegressor(random_state=SEED, n_jobs=1, verbosity=0),
}
GRID_REG = {
    "Ridge": {"m__alpha": [0.001, 0.01, 0.1, 1, 10, 100, 1000]},
    "Lasso": {"m__alpha": [1e-5, 1e-4, 1e-3, 1e-2]},
    "KNN": {"m__n_neighbors": [5, 15, 25, 51, 101], "m__weights": ["uniform", "distance"]},
    "SVR RBF": {"m__C": [0.1, 1, 10], "m__epsilon": [0.005, 0.02], "m__gamma": ["scale", 0.1]},
    "Kernel Ridge RBF": {"m__alpha": [0.1, 1, 10], "m__gamma": [0.01, 0.1, 1]},
    "Random Forest": {"m__max_depth": [4, 8, None], "m__min_samples_leaf": [5, 20, 50], "m__max_features": [1.0, 0.5]},
    "XGBoost": {"m__n_estimators": [200, 500], "m__max_depth": [2, 4, 6], "m__learning_rate": [0.03, 0.1, 0.3]},
}
print({k: int(np.prod([len(v) for v in g.values()])) for k, g in GRID_REG.items()})

{'Ridge': 7, 'Lasso': 4, 'KNN': 10, 'SVR RBF': 12, 'Kernel Ridge RBF': 9, 'Random Forest': 18, 'XGBoost': 18}


## 3. Modelo propuesto

El FC centrado de la planta $p$ el día $t$ se predice como
$\hat{\tilde y}_{pt} = \ell_{pt} + \beta_0 + (\boldsymbol\beta + \mathbf b_p)^\top \mathbf z_{pt}$, donde $\mathbf z_{pt}$ son las covariables
centradas y estandarizadas.

- $\boldsymbol\beta$ es la pendiente común, estimada con Ridge sobre el pasado de las plantas de entrenamiento.
- $\mathbf b_p$ es la desviación de la planta, estimada con Ridge sobre los residuos de su propio pasado. La penalización la encoge
  hacia la pendiente común cuando la planta tiene pocos días o es ruidosa.
- $\ell_{pt}$ es el nivel adaptativo: un promedio móvil exponencial de los residuos observados hasta el día $t-1$, con vida media $h$.
  Corrige cambios de nivel que el pasado no anticipa (degradación, ampliaciones, cambios de operación).

Con $\mathbf b_p = 0$ y $h=\infty$ el modelo se reduce a Ridge. Todo se resuelve en forma cerrada.

In [3]:
class RidgeJerarquico:
    """Pendiente común + desviación por planta encogida hacia la común + nivel adaptativo (EWMA de residuos).
    `lineal="lasso"` cambia la parte común por Lasso; se usa solo en la comparación justa."""

    def __init__(self, alpha_comun=100.0, alpha_planta=100.0, vida_media=30.0, retraso=1, por_planta=True, lineal="ridge"):
        self.alpha_comun, self.alpha_planta, self.vida_media = alpha_comun, alpha_planta, vida_media
        self.retraso, self.por_planta, self.lineal = retraso, por_planta, lineal

    def ajusta(self, X, y):
        """Pendiente común con las filas de entrenamiento (pasado de las otras zonas)."""
        self.esc = StandardScaler().fit(X)
        Z = self.esc.transform(X)
        base = Lasso(alpha=self.alpha_comun, max_iter=10000) if self.lineal == "lasso" else Ridge(alpha=self.alpha_comun)
        self.comun = base.fit(Z, y)
        return self

    def calibra(self, X, y, codigo):
        """Desviación b_p de cada planta, solo con su pasado: Ridge sobre los residuos de la pendiente común."""
        Z = self.esc.transform(X)
        r = y - self.comun.predict(Z)
        self.b = {}
        for c in np.unique(codigo):
            m = codigo == c
            if self.por_planta:
                Zc = Z[m]
                self.b[c] = np.linalg.solve(Zc.T @ Zc + self.alpha_planta * np.eye(Z.shape[1]), Zc.T @ r[m])
            else:
                self.b[c] = np.zeros(Z.shape[1])
        return self

    def clima(self, X, codigo):
        Z = self.esc.transform(X)
        B = np.vstack([self.b[c] for c in codigo])
        return self.comun.predict(Z) + (Z * B).sum(axis=1)

    def nivel(self, y, parte_clima, codigo):
        """Nivel adaptativo: EWMA por planta de los residuos observados hasta t - retraso (filas en orden cronológico)."""
        out = np.zeros(len(y))
        if not np.isfinite(self.vida_media):
            return out
        a = 1.0 - 2.0 ** (-1.0 / self.vida_media)
        r = y - parte_clima
        for c in np.unique(codigo):
            idx = np.flatnonzero(codigo == c)
            ew = np.zeros(len(idx) + 1)
            for k, i in enumerate(idx):
                ew[k + 1] = (1 - a) * ew[k] + a * r[i]
            out[idx] = ew[np.maximum(np.arange(len(idx)) + 1 - self.retraso, 0)]
        return out

    def predice(self, X, y, codigo):
        """Predicción para las filas de una o más plantas en orden cronológico; y solo entra al nivel con retraso."""
        cl = self.clima(X, codigo)
        return cl + self.nivel(y, cl, codigo)


def _corre_propuesto(params, X, y, cod, ent, cal, objetivo):
    """Ajusta con `ent`, calibra b_p con `cal` y predice todas las filas de las plantas de `objetivo`."""
    mod = RidgeJerarquico(**params).ajusta(X[ent], y[ent])
    plantas = np.isin(cod, np.unique(cod[objetivo]))
    mod.calibra(X[cal & plantas], y[cal & plantas], cod[cal & plantas])
    pred = np.full(len(y), np.nan)
    pred[plantas] = mod.predice(X[plantas], y[plantas], cod[plantas])
    return mod, pred


def _puntaje_interno(params, X, y, cod, grp, ent):
    """Bucle interno: deja fuera una zona de entrenamiento; su pasado se parte otra vez en mitad para calibrar y validar."""
    puntajes = []
    for z in np.unique(grp[ent]):
        fit = ent & (grp != z)
        zona = ent & (grp == z)
        orden = pd.Series(np.flatnonzero(zona)).groupby(cod[zona]).cumcount().values
        n_planta = pd.Series(cod[zona]).map(pd.Series(cod[zona]).value_counts()).values
        cal = np.zeros(len(y), bool); val = np.zeros(len(y), bool)
        cal[np.flatnonzero(zona)[orden < n_planta // 2]] = True
        val[np.flatnonzero(zona)[orden >= n_planta // 2]] = True
        _, pred = _corre_propuesto(params, X, y, cod, fit, cal, zona)
        puntajes.append(r2_score(y[val], pred[val]))
    return float(np.mean(puntajes))


def rejilla(grid):
    claves = list(grid)
    return [dict(zip(claves, v)) for v in product(*[grid[k] for k in claves])]


GRID_PROP = {"alpha_comun": [10.0, 100.0], "alpha_planta": [30.0, 100.0, 300.0], "vida_media": [7.0, 14.0, 30.0]}
print("combinaciones del modelo propuesto:", len(rejilla(GRID_PROP)))

combinaciones del modelo propuesto: 18


## 4. Evaluación con el mismo protocolo para todos

Métrica principal: $R^2$ dentro de cada planta, resumido con la mediana, la media, la desviación, el mínimo y el máximo entre las
16 plantas. Además: RMSE, MAE, MAE relativo al FC medio de la planta y MAPE sobre el FC sin centrar (días con FC ≥ 0.05, porque el
MAPE se dispara con FC cercano a 0).

In [4]:
def r2_planta(p):
    return p.groupby("code").apply(lambda g: r2_score(g.fc_c, g.pred))


def mide(p):
    r2 = r2_planta(p)
    fc_hat = p.fc - p.fc_c + p.pred
    m = p.fc >= 0.05
    return {"r2_mediana": r2.median(), "r2_media": r2.mean(), "r2_de": r2.std(), "r2_min": r2.min(), "r2_max": r2.max(),
            "rmse": np.sqrt(mean_squared_error(p.fc_c, p.pred)), "mae": np.abs(p.fc_c - p.pred).mean(),
            "mae_rel_medio": p.groupby("code").apply(lambda g: np.abs(g.fc_c - g.pred).mean() / g.fc.mean()).mean(),
            "mape": float(np.mean(np.abs(p.fc[m] - fc_hat[m]) / p.fc[m]))}


def evalua_base(nombre, feats, grid=None, modelo=None):
    modelo = modelo if modelo is not None else REG[nombre]
    cols = [v + "_c" for v in feats]
    piezas, elegidos = [], []
    for z, ent, pru in particiones_zona():
        est = pipe(clone(modelo))
        if grid:   # validación anidada: el bucle interno deja fuera una zona de entrenamiento a la vez
            est = GridSearchCV(est, grid, cv=LeaveOneGroupOut(), scoring="r2", n_jobs=-1, refit=True)
            est.fit(D.loc[ent, cols], D.fc_c[ent], groups=D.loc[ent, "grupo"])
            elegidos.append({k.replace("m__", ""): v for k, v in est.best_params_.items()})
        else:
            est.fit(D.loc[ent, cols], D.fc_c[ent])
        piezas.append(D.loc[pru, ["code", "date", "fc_c", "fc"]].assign(pred=est.predict(D.loc[pru, cols])))
    return pd.concat(piezas), elegidos


def evalua_propuesto(feats, grid=GRID_PROP, fijos=None, filas_ent=None):
    """Validación anidada del modelo propuesto. `fijos` salta la búsqueda; `filas_ent` permite submuestrear el entrenamiento."""
    cols = [v + "_c" for v in feats]
    X, y, cod, grp = D[cols].values, D.fc_c.values, D.code.values, D.grupo.values
    piezas, elegidos = [], []
    for z, ent, pru in particiones_zona():
        if filas_ent is not None:
            ent = ent & filas_ent
        if fijos is None:
            cands = rejilla(grid)
            pts = Parallel(n_jobs=-1)(delayed(_puntaje_interno)(c, X, y, cod, grp, ent) for c in cands)
            mejor = cands[int(np.argmax(pts))]
        else:
            mejor = fijos
        elegidos.append(mejor)
        _, pred = _corre_propuesto(mejor, X, y, cod, ent, PASADO, pru)
        piezas.append(D.loc[pru, ["code", "date", "fc_c", "fc"]].assign(pred=pred[pru]))
    return pd.concat(piezas), elegidos


NOMBRE_PROP = "Ridge jerárquico (propuesto)"

## 5. Resultados

In [5]:
filas, preds, params = [], {}, {}
for ajustar in (False, True):
    for nombre in REG:
        t0 = time.time()
        p, el = evalua_base(nombre, VARS, GRID_REG.get(nombre) if ajustar else None)
        filas.append({"modelo": nombre, "grupo": "clasico", "ajuste": "rejilla" if ajustar else "defecto", **mide(p),
                      "segundos": time.time() - t0})
        if ajustar:
            preds[nombre], params[nombre] = p, el

p = D.loc[D.prueba_t, ["code", "date", "fc_c", "fc"]].assign(pred=D.loc[D.prueba_t, "fcayer_c"])
filas.append({"modelo": "Referencia: persistencia", "grupo": "referencia", "ajuste": "defecto", **mide(p), "segundos": 0.0})
preds["Referencia: persistencia"] = p

t0 = time.time()
p_prop, el_prop = evalua_propuesto(VARS)
seg_prop = time.time() - t0
filas.append({"modelo": NOMBRE_PROP, "grupo": "propuesto", "ajuste": "rejilla", **mide(p_prop), "segundos": seg_prop})
preds[NOMBRE_PROP], params[NOMBRE_PROP] = p_prop, el_prop

### Comparación con la misma información

El modelo propuesto usa la generación observada hasta el día anterior, que los modelos de referencia no reciben. Para separar el aporte de
esa información del aporte del modelo, Lasso se repite con la misma información, de dos formas:

- **Lasso + FC de ayer**: el FC del día anterior entra como una variable más.
- **Lasso + nivel adaptativo**: Lasso como parte común y el mismo nivel adaptativo del modelo propuesto, sin desviaciones por planta.

In [6]:
t0 = time.time()
p, el = evalua_base("Lasso + FC de ayer", VARS + ["fcayer"], GRID_REG["Lasso"], modelo=REG["Lasso"])
filas.append({"modelo": "Lasso + FC de ayer", "grupo": "misma informacion", "ajuste": "rejilla", **mide(p), "segundos": time.time() - t0})
preds["Lasso + FC de ayer"], params["Lasso + FC de ayer"] = p, el

t0 = time.time()
grid_lasso_nivel = {"alpha_comun": [1e-4, 1e-3], "vida_media": [7.0, 14.0, 30.0], "por_planta": [False], "lineal": ["lasso"]}
p, el = evalua_propuesto(VARS, grid=grid_lasso_nivel)
filas.append({"modelo": "Lasso + nivel adaptativo", "grupo": "misma informacion", "ajuste": "rejilla", **mide(p),
              "segundos": time.time() - t0})
preds["Lasso + nivel adaptativo"], params["Lasso + nivel adaptativo"] = p, el

res = pd.DataFrame(filas)
res.to_csv(RES / "regresion.csv", index=False)
aj = res[res.ajuste == "rejilla"].set_index("modelo").sort_values("r2_mediana", ascending=False)
display(aj[["grupo", "r2_mediana", "r2_media", "r2_de", "r2_min", "rmse", "mae_rel_medio", "mape", "segundos"]].round(4))
display(res[res.ajuste == "defecto"].set_index("modelo")[["r2_mediana", "mae_rel_medio", "segundos"]].round(4))

,grupo,r2_mediana,r2_media,r2_de,r2_min,rmse,mae_rel_medio,mape,segundos
modelo,,,,,,,,,
Ridge jerárquico (propuesto),propuesto,0.5245,0.5047,0.0911,0.3027,0.0416,0.1255,0.1469,1.7496
Lasso + nivel adaptativo,misma informacion,0.5219,0.4850,0.1066,0.1855,0.0423,0.1287,0.1505,0.7001
Lasso + FC de ayer,misma informacion,0.4901,0.4057,0.2334,-0.3163,0.0450,0.1382,0.1614,0.5508
Lasso,clasico,0.4685,0.3862,0.2671,-0.4623,0.0455,0.1402,0.1634,0.7993
Regresión lineal,clasico,0.4661,0.3814,0.2710,-0.4687,0.0456,0.1405,0.1634,0.0759
XGBoost,clasico,0.4631,0.3679,0.2650,-0.4626,0.0460,0.1425,0.1668,39.3096
Ridge,clasico,0.4611,0.3825,0.2687,-0.4733,0.0457,0.1407,0.1640,10.0143
Kernel Ridge RBF,clasico,0.4332,0.3758,0.2478,-0.4285,0.0461,0.1426,0.1678,73.9608
Random Forest,clasico,0.4307,0.3662,0.2630,-0.4887,0.0463,0.1439,0.1694,161.7434


,r2_mediana,mae_rel_medio,segundos
modelo,,,
Referencia: nivel de la planta,-0.0392,0.1957,0.0829
Regresión lineal,0.4661,0.1405,0.1031
Ridge,0.4661,0.1405,0.1257
Lasso,0.4685,0.1404,0.1004
KNN,0.3959,0.1504,0.1908
SVR RBF,0.3582,0.1516,5.7645
Kernel Ridge RBF,0.4101,0.1465,5.6884
Random Forest,0.4204,0.1483,32.2461
XGBoost,0.3455,0.1622,1.0624


In [7]:
for nombre, el in params.items():
    if el:
        print(f"{nombre}: " + " | ".join(str(e) for e in el))

Ridge: {'alpha': 100} | {'alpha': 100} | {'alpha': 100} | {'alpha': 100} | {'alpha': 100} | {'alpha': 100}
Lasso: {'alpha': 0.001} | {'alpha': 0.0001} | {'alpha': 0.001} | {'alpha': 0.001} | {'alpha': 0.001} | {'alpha': 0.001}
KNN: {'n_neighbors': 101, 'weights': 'uniform'} | {'n_neighbors': 101, 'weights': 'distance'} | {'n_neighbors': 101, 'weights': 'uniform'} | {'n_neighbors': 101, 'weights': 'uniform'} | {'n_neighbors': 101, 'weights': 'uniform'} | {'n_neighbors': 101, 'weights': 'uniform'}
SVR RBF: {'C': 0.1, 'epsilon': 0.02, 'gamma': 0.1} | {'C': 0.1, 'epsilon': 0.02, 'gamma': 0.1} | {'C': 0.1, 'epsilon': 0.02, 'gamma': 0.1} | {'C': 0.1, 'epsilon': 0.02, 'gamma': 0.1} | {'C': 0.1, 'epsilon': 0.02, 'gamma': 0.1} | {'C': 0.1, 'epsilon': 0.02, 'gamma': 0.1}
Kernel Ridge RBF: {'alpha': 1, 'gamma': 0.01} | {'alpha': 0.1, 'gamma': 0.01} | {'alpha': 10, 'gamma': 0.01} | {'alpha': 10, 'gamma': 0.01} | {'alpha': 1, 'gamma': 0.01} | {'alpha': 10, 'gamma': 0.01}
Random Forest: {'max_depth'

### $R^2$ por planta

In [8]:
r2p = pd.DataFrame({m: r2_planta(preds[m]) for m in preds})
r2p.index = r2p.index.map(tabla.drop_duplicates("code").set_index("code").planta)
r2p.to_csv(RES / "r2_por_planta.csv")
r2p[[NOMBRE_PROP, "Lasso", "Lasso + nivel adaptativo", "Lasso + FC de ayer", "XGBoost"]].sort_values(NOMBRE_PROP).round(3)

,Ridge jerárquico (propuesto),Lasso,Lasso + nivel adaptativo,Lasso + FC de ayer,XGBoost
code,,,,,
SUNNORTE,0.303,0.043,0.186,0.106,-0.001
SOLAR PLANETA RICA,0.355,0.333,0.361,0.328,0.325
PORTON DEL SOL,0.417,0.311,0.422,0.327,0.312
PARQUE SOLAR TEPUY,0.457,0.382,0.451,0.389,0.382
BSB 503,0.461,0.371,0.416,0.377,0.330
BSB 504,0.469,0.378,0.428,0.381,0.328
FUNDACION,0.501,0.510,0.508,0.519,0.506
GUAYEPO,0.515,0.427,0.511,0.461,0.420
EL PASO,0.534,0.517,0.533,0.536,0.512


## 6. Pruebas estadísticas

- Intervalo de confianza del 95 % de la mediana de $R^2_p$ con bootstrap por plantas (5,000 réplicas).
- Wilcoxon de rangos con signo sobre las 16 diferencias pareadas de $R^2_p$.
- Diebold-Mariano por planta (pérdida cuadrática, varianza de Newey-West), para comparar el error diario de dos modelos de regresión.

In [9]:
rng = np.random.default_rng(SEED)


def ic_mediana(v, reps=5000):
    v = np.asarray(v)
    meds = np.median(v[rng.integers(0, len(v), size=(reps, len(v)))], axis=1)
    return np.percentile(meds, [2.5, 97.5])


def diebold_mariano(e1, e2):
    """DM con pérdida cuadrática y varianza HAC de Newey-West (rezago floor(T^(1/3)))."""
    d = e1 ** 2 - e2 ** 2
    T = len(d); L = int(np.floor(T ** (1 / 3)))
    dc = d - d.mean()
    s = dc @ dc / T
    for k in range(1, L + 1):
        s += 2 * (1 - k / (L + 1)) * (dc[k:] @ dc[:-k]) / T
    est = d.mean() / np.sqrt(s / T)
    return est, 2 * stats.norm.sf(abs(est))


filas = []
for m in preds:
    if m == NOMBRE_PROP:
        continue
    lo, hi = ic_mediana(r2p[m])
    dif = r2p[NOMBRE_PROP] - r2p[m]
    w = stats.wilcoxon(dif) if np.any(dif != 0) else None
    mejor = peor = 0
    for c in tabla.code.unique():
        a = preds[NOMBRE_PROP].query("code == @c").sort_values("date")
        b = preds[m].query("code == @c").sort_values("date")
        est, pv = diebold_mariano((a.fc_c - a.pred).values, (b.fc_c - b.pred).values)
        mejor += (pv < 0.05) and est < 0
        peor += (pv < 0.05) and est > 0
    filas.append({"modelo": m, "r2_mediana": r2p[m].median(), "ic95_lo": lo, "ic95_hi": hi,
                  "dif_mediana_vs_propuesto": dif.median(), "plantas_mejor_propuesto": int((dif > 0).sum()),
                  "wilcoxon_p": w.pvalue if w else np.nan,
                  "dm_plantas_mejor_propuesto": mejor, "dm_plantas_mejor_referencia": peor})
lo, hi = ic_mediana(r2p[NOMBRE_PROP])
filas.insert(0, {"modelo": NOMBRE_PROP, "r2_mediana": r2p[NOMBRE_PROP].median(), "ic95_lo": lo, "ic95_hi": hi})
pruebas = pd.DataFrame(filas)
pruebas.to_csv(RES / "pruebas_regresion.csv", index=False)
pruebas.round(4)

,modelo,r2_mediana,ic95_lo,ic95_hi,dif_mediana_vs_propuesto,plantas_mejor_propuesto,wilcoxon_p,dm_plantas_mejor_propuesto,dm_plantas_mejor_referencia
0,Ridge jerárquico (propuesto),0.5245,0.4609,0.5645,NaN,NaN,NaN,NaN,NaN
1,Referencia: nivel de la planta,-0.0392,-0.0580,-0.0144,0.5431,16.0,0.0000,16.0,0.0
2,Regresión lineal,0.4661,0.3437,0.5464,0.0281,15.0,0.0001,6.0,0.0
3,Ridge,0.4611,0.3571,0.5401,0.0319,15.0,0.0001,8.0,0.0
4,Lasso,0.4685,0.3715,0.5449,0.0289,15.0,0.0001,6.0,0.0
5,KNN,0.4139,0.3088,0.4937,0.0953,16.0,0.0000,13.0,0.0
6,SVR RBF,0.4066,0.2895,0.5020,0.0869,16.0,0.0000,11.0,0.0
7,Kernel Ridge RBF,0.4332,0.3563,0.5168,0.0561,16.0,0.0000,9.0,0.0
8,Random Forest,0.4307,0.3433,0.5197,0.0701,16.0,0.0000,10.0,0.0
9,XGBoost,0.4631,0.3284,0.5315,0.0688,15.0,0.0001,8.0,0.0


## 7. Ablación

Cada variante cambia un solo componente y repite la búsqueda en su propia rejilla con el mismo protocolo.

In [10]:
VARIANTES = {
    "Modelo completo": dict(feats=VARS, grid=GRID_PROP),
    "Sin desviaciones por planta": dict(feats=VARS, grid={**GRID_PROP, "alpha_planta": [1.0], "por_planta": [False]}),
    "Sin nivel adaptativo": dict(feats=VARS, grid={**GRID_PROP, "vida_media": [np.inf]}),
    "Sin ambos (Ridge)": dict(feats=VARS, grid={**GRID_PROP, "alpha_planta": [1.0], "por_planta": [False], "vida_media": [np.inf]}),
    "Nivel con 7 días de retraso": dict(feats=VARS, grid={**GRID_PROP, "retraso": [7]}),
    "Solo radiación": dict(feats=["rad_bm"], grid=GRID_PROP),
    "Radiación y nubosidad de ERA5": dict(feats=["rad_e5", "nube_e5", "temp_bm", "humedad_bm", "viento_bm"], grid=GRID_PROP),
    "Índice físico en lugar de radiación": dict(feats=["e_fis_bm", "nube_bm", "temp_bm", "humedad_bm", "viento_bm"], grid=GRID_PROP),
}
filas, r2_abl = [], {}
for nombre, cfg in VARIANTES.items():
    p = p_prop if nombre == "Modelo completo" else evalua_propuesto(cfg["feats"], grid=cfg["grid"])[0]
    r2_abl[nombre] = r2_planta(p)
    dif = r2_abl[nombre] - r2_abl["Modelo completo"]
    w = stats.wilcoxon(dif).pvalue if np.any(dif != 0) else np.nan
    filas.append({"variante": nombre, **{k: v for k, v in mide(p).items() if k in ("r2_mediana", "r2_media", "r2_min", "mae_rel_medio")},
                  "dif_r2_mediana": dif.median(), "wilcoxon_p": w})
ablacion = pd.DataFrame(filas)
ablacion.to_csv(RES / "ablacion.csv", index=False)
ablacion.round(4)

,variante,r2_mediana,r2_media,r2_min,mae_rel_medio,dif_r2_mediana,wilcoxon_p
0,Modelo completo,0.5245,0.5047,0.3027,0.1255,0.0000,NaN
1,Sin desviaciones por planta,0.5194,0.4836,0.1863,0.1287,-0.0077,0.0092
2,Sin nivel adaptativo,0.4857,0.4173,-0.4334,0.1356,-0.0134,0.0155
3,Sin ambos (Ridge),0.4657,0.3810,-0.4733,0.1406,-0.0284,0.0001
4,Nivel con 7 días de retraso,0.5152,0.4924,0.2291,0.1273,-0.0008,0.1439
5,Solo radiación,0.5152,0.4844,0.2674,0.1295,-0.0215,0.0004
6,Radiación y nubosidad de ERA5,0.4789,0.4557,0.2810,0.1340,-0.0398,0.0004
7,Índice físico en lugar de radiación,0.5247,0.5050,0.2987,0.1255,0.0003,0.4332


## 8. ¿Hay sobreajuste?

Dos pruebas: una de permutación y una curva de aprendizaje. En la permutación se barajan los FC del pasado
dentro de cada planta, con lo que la parte común y las desviaciones por planta ya no pueden aprender del clima; el nivel adaptativo
sigue usando la generación observada. La curva de aprendizaje usa fracciones del pasado de entrenamiento. Ambas usan los
hiperparámetros más elegidos en la validación anidada.

In [11]:
moda = lambda els: pd.Series([json.dumps(e, sort_keys=True, default=float) for e in els]).mode()[0]
fijos_prop = json.loads(moda(el_prop))
fijos_xgb = json.loads(moda(params["XGBoost"]))
fijos_lasso = json.loads(moda(params["Lasso"]))
print("propuesto:", fijos_prop, "| XGBoost:", fijos_xgb, "| Lasso:", fijos_lasso)

obs = r2_planta(evalua_propuesto(VARS, fijos=fijos_prop)[0]).median()
cols = [v + "_c" for v in VARS]
X0, y0, cod0 = D[cols].values, D.fc_c.values, D.code.values


def r2_permutado(semilla):
    """Baraja el FC del pasado dentro de cada planta (entrenamiento de la parte común y calibración de b_p).
    El nivel adaptativo sigue usando la generación observada, así que el nulo mide lo que se logra sin aprender del clima."""
    g = np.random.default_rng(semilla)
    yp = y0.copy()
    for c in np.unique(cod0):
        idx = np.flatnonzero(PASADO & (cod0 == c))
        yp[idx] = yp[g.permutation(idx)]
    piezas = []
    for z, ent, pru in particiones_zona():
        mod = RidgeJerarquico(**fijos_prop).ajusta(X0[ent], yp[ent])
        plantas = np.isin(cod0, np.unique(cod0[pru]))
        mod.calibra(X0[PASADO & plantas], yp[PASADO & plantas], cod0[PASADO & plantas])
        pred = np.full(len(y0), np.nan)
        pred[plantas] = mod.predice(X0[plantas], y0[plantas], cod0[plantas])
        piezas.append(D.loc[pru, ["code", "fc_c"]].assign(pred=pred[pru]))
    return r2_planta(pd.concat(piezas)).median()


N_PERM = 200
nulos = np.array(Parallel(n_jobs=-1)(delayed(r2_permutado)(s) for s in range(N_PERM)))
p_perm = (1 + np.sum(nulos >= obs)) / (1 + N_PERM)
pd.DataFrame({"r2_nulo": nulos}).to_csv(RES / "permutacion.csv", index=False)
print(f"R² observado {obs:.3f} | nulo: mediana {np.median(nulos):.3f}, máximo {nulos.max():.3f} | p = {p_perm:.4f}")

propuesto: {'alpha_comun': 10.0, 'alpha_planta': 300.0, 'vida_media': 7.0} | XGBoost: {'learning_rate': 0.03, 'max_depth': 2, 'n_estimators': 200} | Lasso: {'alpha': 0.001}


R² observado 0.523 | nulo: mediana 0.038, máximo 0.096 | p = 0.0050


In [12]:
FRACS = [0.1, 0.25, 0.5, 0.75, 1.0]
filas = []
for f in FRACS:
    sub = np.random.default_rng(SEED).random(len(D)) < f
    pp, _ = evalua_propuesto(VARS, fijos=fijos_prop, filas_ent=sub)
    filas.append({"fraccion": f, "modelo": NOMBRE_PROP, "r2_mediana": r2_planta(pp).median()})
    for nombre, fijo in [("Lasso", fijos_lasso), ("XGBoost", fijos_xgb)]:
        piezas = []
        for z, ent, pru in particiones_zona():
            est = pipe(clone(REG[nombre]).set_params(**fijo)).fit(D.loc[ent & sub, cols], D.fc_c[ent & sub])
            piezas.append(D.loc[pru, ["code", "fc_c"]].assign(pred=est.predict(D.loc[pru, cols])))
        filas.append({"fraccion": f, "modelo": nombre, "r2_mediana": r2_planta(pd.concat(piezas)).median()})
curva = pd.DataFrame(filas)
curva.to_csv(RES / "curva_aprendizaje.csv", index=False)
curva.pivot(index="fraccion", columns="modelo", values="r2_mediana").round(3)

modelo,Lasso,Ridge jerárquico (propuesto),XGBoost
fraccion,,,
0.10,0.446,0.521,0.393
0.25,0.451,0.519,0.443
0.50,0.454,0.522,0.450
0.75,0.468,0.525,0.471
1.00,0.469,0.523,0.463


## 9. Partición aleatoria frente a zona fuera

Los modelos de referencia, con valores por defecto, se repiten con una partición aleatoria de filas en 5 pliegues. Ahí días vecinos de
una misma planta y de plantas cercanas caen a ambos lados, y el nivel de cada planta se calcula con días de todo el periodo.
La diferencia mide cuánto se infla el resultado sin el diseño espacio-temporal.

In [13]:
def r2_aleatoria(nombre):
    cols = [v + "_c" for v in VARS]
    piezas = []
    for tr, te in KFold(5, shuffle=True, random_state=SEED).split(tabla):
        ent = np.zeros(len(tabla), bool); ent[tr] = True
        d = prepara(tabla, ent)
        est = pipe(clone(REG[nombre])).fit(d.loc[ent, cols], d.fc_c[ent])
        piezas.append(d.loc[~ent, ["code", "fc_c", "fc"]].assign(pred=est.predict(d.loc[~ent, cols])))
    return mide(pd.concat(piezas))["r2_mediana"]


defecto = res[res.ajuste == "defecto"].set_index("modelo")
diseno = pd.DataFrame({"zona_fuera": defecto.r2_mediana.drop("Referencia: persistencia"),
                       "aleatoria": {m: r2_aleatoria(m) for m in REG}})
diseno["inflado"] = diseno.aleatoria - diseno.zona_fuera
diseno.to_csv(RES / "diseno_validacion.csv")
diseno.round(3)

,zona_fuera,aleatoria,inflado
Referencia: nivel de la planta,-0.039,-0.000,0.039
Regresión lineal,0.466,0.495,0.028
Ridge,0.466,0.495,0.028
Lasso,0.469,0.495,0.026
KNN,0.396,0.467,0.071
SVR RBF,0.358,0.479,0.120
Kernel Ridge RBF,0.410,0.501,0.091
Random Forest,0.420,0.487,0.066
XGBoost,0.346,0.445,0.099


## 10. Pendientes por planta y entorno

In [14]:
mod = RidgeJerarquico(**fijos_prop).ajusta(X0[PASADO], y0[PASADO]).calibra(X0[PASADO], y0[PASADO], cod0[PASADO])
escala = mod.esc.scale_
pend = pd.DataFrame({c: (mod.comun.coef_ + mod.b[c]) / escala for c in mod.b}, index=VARS).T
pend.index = pend.index.map(tabla.drop_duplicates("code").set_index("code").planta)
pend["pendiente_comun_rad"] = mod.comun.coef_[0] / escala[0]
pend.to_csv(RES / "pendientes.csv")
print("Pendiente común de radiación:", round(mod.comun.coef_[0] / escala[0], 4), "FC por kWh/m²")
pend.rad_bm.describe().round(4)

Pendiente común de radiación: 0.036 FC por kWh/m²


count    16.0000
mean      0.0358
std       0.0031
min       0.0308
25%       0.0335
50%       0.0365
75%       0.0380
max       0.0403
Name: rad_bm, dtype: float64

In [15]:
entorno = {"python": platform.python_version(), "sklearn": sklearn.__version__, "xgboost": xgboost.__version__,
           "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
           "procesador": platform.processor(), "nucleos": os.cpu_count(), "sistema": platform.platform(),
           "hiperparametros_propuesto": el_prop, "fijos_propuesto": fijos_prop, "permutacion_p": p_perm, "r2_permutacion_obs": obs}
json.dump(entorno, open(RES / "entorno.json", "w"), indent=1, default=str, ensure_ascii=False)
entorno

{'python': '3.10.11',
 'sklearn': '1.7.2',
 'xgboost': '3.2.0',
 'numpy': '2.2.6',
 'pandas': '2.3.3',
 'scipy': '1.15.3',
 'procesador': 'AMD64 Family 23 Model 160 Stepping 0, AuthenticAMD',
 'nucleos': 8,
 'sistema': 'Windows-10-10.0.26300-SP0',
 'hiperparametros_propuesto': [{'alpha_comun': 100.0,
   'alpha_planta': 300.0,
   'vida_media': 14.0},
  {'alpha_comun': 10.0, 'alpha_planta': 300.0, 'vida_media': 7.0},
  {'alpha_comun': 10.0, 'alpha_planta': 100.0, 'vida_media': 14.0},
  {'alpha_comun': 10.0, 'alpha_planta': 300.0, 'vida_media': 7.0},
  {'alpha_comun': 100.0, 'alpha_planta': 100.0, 'vida_media': 14.0},
  {'alpha_comun': 10.0, 'alpha_planta': 100.0, 'vida_media': 7.0}],
 'fijos_propuesto': {'alpha_comun': 10.0,
  'alpha_planta': 300.0,
  'vida_media': 7.0},
 'permutacion_p': np.float64(0.004975124378109453),
 'r2_permutacion_obs': np.float64(0.5231960311069117)}